<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">처음부터 만드는 대형 언어 모델</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 4장 연습 문제 해답

In [ ]:
from importlib.metadata import version

print("torch version:", version("torch"))

# 연습 문제 4.1: 피드포워드와 어텐션 모듈의 파라미터

In [ ]:
from gpt import TransformerBlock

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

block = TransformerBlock(GPT_CONFIG_124M)
print(block)

In [ ]:
total_params = sum(p.numel() for p in block.ff.parameters())
print(f"피드포워드 모듈의 총 파라미터 수: {total_params:,}")

In [ ]:
total_params = sum(p.numel() for p in block.att.parameters())
print(f"어텐션 모듈의 총 파라미터 수: {total_params:,}")

- 위의 결과는 단일 트랜스포머 블록에 대한 것입니다
- 선택적으로 12를 곱하여 1억 2400만 GPT 모델의 모든 트랜스포머 블록을 포함할 수 있습니다

**보너스: 수학적 분석**

- 이러한 파라미터 개수가 수학적으로 어떻게 계산되는지에 관심이 있는 분들을 위해, 아래에 분석을 제공합니다 (`emb_dim=768`로 가정):


피드포워드 모듈:

- 첫 번째 `Linear` 레이어: 768개 입력 × 4×768개 출력 + 4×768개 편향 유닛 = 2,362,368
- 두 번째 `Linear` 레이어: 4×768개 입력 × 768개 출력 + 768개 편향 유닛 = 2,360,064
- 총계: 첫 번째 `Linear` 레이어 + 두 번째 `Linear` 레이어 = 2,362,368 + 2,360,064 = 4,722,432

어텐션 모듈:

- `W_query`: 768개 입력 × 768개 출력 = 589,824 
- `W_key`: 768개 입력 × 768개 출력 = 589,824
- `W_value`: 768개 입력 × 768개 출력 = 589,824 
- `out_proj`: 768개 입력 × 768개 출력 + 768개 편향 유닛 = 590,592
- 총계: `W_query` + `W_key` + `W_value` + `out_proj` = 3×589,824 + 590,592 = 2,360,064

# 연습 문제 4.2: 더 큰 GPT 모델 초기화하기

- **GPT2-small** (우리가 이미 구현한 1억 2400만 구성):
    - "emb_dim" = 768
    - "n_layers" = 12
    - "n_heads" = 12

- **GPT2-medium:**
    - "emb_dim" = 1024
    - "n_layers" = 24
    - "n_heads" = 16

- **GPT2-large:**
    - "emb_dim" = 1280
    - "n_layers" = 36
    - "n_heads" = 20

- **GPT2-XL:**
    - "emb_dim" = 1600
    - "n_layers" = 48
    - "n_heads" = 25

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}


def get_config(base_config, model_name="gpt2-small"):
    GPT_CONFIG = base_config.copy()

    if model_name == "gpt2-small":
        GPT_CONFIG["emb_dim"] = 768
        GPT_CONFIG["n_layers"] = 12
        GPT_CONFIG["n_heads"] = 12

    elif model_name == "gpt2-medium":
        GPT_CONFIG["emb_dim"] = 1024
        GPT_CONFIG["n_layers"] = 24
        GPT_CONFIG["n_heads"] = 16

    elif model_name == "gpt2-large":
        GPT_CONFIG["emb_dim"] = 1280
        GPT_CONFIG["n_layers"] = 36
        GPT_CONFIG["n_heads"] = 20

    elif model_name == "gpt2-xl":
        GPT_CONFIG["emb_dim"] = 1600
        GPT_CONFIG["n_layers"] = 48
        GPT_CONFIG["n_heads"] = 25

    else:
        raise ValueError(f"잘못된 모델 이름 {model_name}")

    return GPT_CONFIG


def calculate_size(model): # 장 코드 기반
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"총 파라미터 수: {total_params:,}")

    total_params_gpt2 =  total_params - sum(p.numel() for p in model.out_head.parameters())
    print(f"가중치 타이잉을 고려한 훈련 가능한 파라미터 수: {total_params_gpt2:,}")
    
    # 총 크기를 바이트로 계산 (float32로 가정, 파라미터당 4바이트)
    total_size_bytes = total_params * 4
    
    # 메가바이트로 변환
    total_size_mb = total_size_bytes / (1024 * 1024)
    
    print(f"모델의 총 크기: {total_size_mb:.2f} MB")

In [ ]:
from gpt import GPTModel


for model_abbrev in ("small", "medium", "large", "xl"):
    model_name = f"gpt2-{model_abbrev}"
    CONFIG = get_config(GPT_CONFIG_124M, model_name=model_name)
    model = GPTModel(CONFIG)
    print(f"\n\n{model_name}:")
    calculate_size(model)

# 연습 문제 4.3: 별도의 드롭아웃 파라미터 사용하기

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate_emb": 0.1,        # 새로 추가: 임베딩 레이어용 드롭아웃
    "drop_rate_attn": 0.1,       # 새로 추가: 멀티헤드 어텐션용 드롭아웃  
    "drop_rate_shortcut": 0.1,   # 새로 추가: 숏컷 연결용 드롭아웃
    "qkv_bias": False
}

In [ ]:
import torch.nn as nn
from gpt import MultiHeadAttention, LayerNorm, FeedForward


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate_attn"], # 새로 추가: 멀티헤드 어텐션용 드롭아웃
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate_shortcut"])

    def forward(self, x):
        # 어텐션 블록에 대한 숏컷 연결
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # 형태 [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력을 다시 더합니다

        # 피드포워드 블록에 대한 숏컷 연결
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # 원본 입력을 다시 더합니다

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate_emb"]) # 새로 추가: 임베딩 레이어용 드롭아웃

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # 형태 [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
import torch

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)